# Day 1 – Data Understanding & Exploratory Data Analysis

**Predictive Denial Prevention AI Platform**

> **Important Framing:** Due to dataset constraints, Provider-level `PotentialFraud` is used as a proxy label for Claim Anomaly & Audit Risk. This is explicitly documented per the corporate engineering decision.

### Notebook Objectives

1. Download & ingest the Kaggle Healthcare Fraud dataset via `kagglehub`
2. Inspect the schema & structure of every source file
3. Build the data merging pipeline (Union → Left Join → Left Join)
4. Profile missing values, numerical distributions, categorical frequencies
5. Analyse class imbalance, chronic-condition correlations, provider-level patterns, and temporal trends
6. Summarise key findings that feed into Day 2 (Feature Engineering)


In [0]:
# ============================================================
# CELL 1 – Install kagglehub & download dataset to Volume
# ============================================================
# Uncomment the next two lines on first run inside Databricks:
# %pip install kagglehub
# dbutils.library.restartPython()

import kagglehub
import shutil
import os
import glob

# Target path – Databricks Unity Catalog Volume
VOLUME_PATH = "/Volumes/capstone/datasets/datasets/"
if not os.path.exists("/Volumes"):          # local dev fallback
    VOLUME_PATH = os.path.join(os.path.dirname(os.getcwd()), "data")

os.makedirs(VOLUME_PATH, exist_ok=True)

path = kagglehub.dataset_download(
    "rohitrox/healthcare-provider-fraud-detection-analysis")
print(f"Ephemeral download path: {path}")

for f in os.listdir(path):
    src, dst = os.path.join(path, f), os.path.join(VOLUME_PATH, f)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print(f"Files in Volume ({VOLUME_PATH}):")
for f in sorted(os.listdir(VOLUME_PATH)):
    print(f"  {f}  ({os.path.getsize(os.path.join(VOLUME_PATH, f)) / 1e6:.2f} MB)")

In [0]:
# ============================================================
# CELL 2 – PySpark initialisation & smart file discovery
# ============================================================
import os
import glob as _glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, count, isnan, isnull, sum as _sum,
    avg, round as _round, datediff, to_date, desc, countDistinct,
    month, year, max as _max, min as _min
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

spark = SparkSession.builder.appName("PredictiveDenialEDA").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

VOLUME_PATH = "/Volumes/capstone/datasets/datasets/"


def find_file(keyword, prefix="Train"):
    """
    Uses os.listdir + startswith for exact prefix matching.
    Avoids glob unreliability on Databricks Unity Catalog FUSE mounts.
    """
    matches = [
        os.path.join(VOLUME_PATH, f)
        for f in os.listdir(VOLUME_PATH)
        if f.startswith(prefix) and keyword in f
    ]
    assert matches, f"No file found for prefix='{prefix}' keyword='{keyword}' in {VOLUME_PATH}"
    assert len(matches) == 1, f"Ambiguous match for '{keyword}': {matches}"
    return matches[0]


# → Train_Inpatientdata-...csv
INPATIENT_PATH = find_file("Inpatientdata")
# → Train_Outpatientdata-...csv
OUTPATIENT_PATH = find_file("Outpatientdata")
# → Train_Beneficiarydata-...csv
BENEFICIARY_PATH = find_file("Beneficiarydata")
TRAIN_LABEL_PATH = find_file("", prefix="Train-")   # → Train-1542865627584.csv

print("Resolved file paths:")
for label, p in [("Inpatient",   INPATIENT_PATH),
                 ("Outpatient",  OUTPATIENT_PATH),
                 ("Beneficiary", BENEFICIARY_PATH),
                 ("Labels",      TRAIN_LABEL_PATH)]:
    print(f"  {label:12s} → {os.path.basename(p)}")

---

## Section 1 – Understand Claim Structure

Load each source file individually and inspect its schema, shape, and sample rows.


In [0]:
# ============================================================
# CELL 3 – Load raw datasets & inspect schemas
# ============================================================

# Add nullValue="NA" to cleanly cast 'NA' strings to actual SQL Nulls
inpatient_raw = spark.read.csv(
    INPATIENT_PATH, header=True, inferSchema=True, nullValue="NA")
outpatient_raw = spark.read.csv(
    OUTPATIENT_PATH, header=True, inferSchema=True, nullValue="NA")
beneficiary_raw = spark.read.csv(
    BENEFICIARY_PATH, header=True, inferSchema=True, nullValue="NA")
train_labels = spark.read.csv(
    TRAIN_LABEL_PATH, header=True, inferSchema=True, nullValue="NA")

datasets = {
    "Inpatient Claims":  inpatient_raw,
    "Outpatient Claims": outpatient_raw,
    "Beneficiary":       beneficiary_raw,
    "Train Labels":      train_labels,
}

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  {name}  |  Rows: {df.count():,}  |  Columns: {len(df.columns)}")
    print(f"{'='*60}")
    df.printSchema()

In [0]:
# ============================================================
# CELL 4 – Sample rows from each dataset
# ============================================================
for name, df in datasets.items():
    print(f"\n── {name} (first 3 rows) ──")
    df.show(3, truncate=False)

---

## Section 2 – Individual Dataset Profiling

Summary statistics, date ranges, and key characteristics for each source table.


In [0]:
# ============================================================
# CELL 5 – Inpatient Claims profiling
# ============================================================
print("Inpatient Claims - Summary Statistics (Numerical Columns)")
# inpatient_raw.describe().show()

# Admission duration (inpatient only)
inp_dates = inpatient_raw.withColumn(
    "AdmissionDt_d", to_date(col("AdmissionDt"), "yyyy-MM-dd")
).withColumn(
    "DischargeDt_d", to_date(col("DischargeDt"), "yyyy-MM-dd")
).withColumn(
    "AdmissionDuration", datediff(col("DischargeDt_d"), col("AdmissionDt_d"))
)

print("\nAdmission Duration Stats (days):")
inp_dates.select("AdmissionDuration").describe().show()

print(
    f"Unique Attending Physicians : {inpatient_raw.select('AttendingPhysician').distinct().count():,}")
print(
    f"Unique Operating Physicians: {inpatient_raw.select('OperatingPhysician').distinct().count():,}")
print(
    f"Unique Providers           : {inpatient_raw.select('Provider').distinct().count():,}")
print(
    f"Unique Beneficiaries       : {inpatient_raw.select('BeneID').distinct().count():,}")

In [0]:
# ============================================================
# CELL 6 – Outpatient Claims profiling
# ============================================================
print("Outpatient Claims - Summary Statistics")
# outpatient_raw.describe().show()
print(
    f"Unique Providers     : {outpatient_raw.select('Provider').distinct().count():,}")
print(
    f"Unique Beneficiaries : {outpatient_raw.select('BeneID').distinct().count():,}")

In [0]:
# ============================================================
# CELL 7 – Beneficiary profiling
# ============================================================
print("Beneficiary - Summary Statistics")
# beneficiary_raw.describe().show()

# Chronic condition columns
chronic_cols = [
    c for c in beneficiary_raw.columns if c.startswith("ChronicCond_")]
print(f"\nChronic Condition Flags ({len(chronic_cols)}):")
for c in chronic_cols:
    dist = beneficiary_raw.groupBy(c).count().orderBy(c).toPandas()
    vals = ", ".join(
        [f"{row[c]}={row['count']:,}" for _, row in dist.iterrows()])
    print(f"  {c}: {vals}")

In [0]:
# ============================================================
# CELL 8 – Train Labels profiling
# ============================================================
print("Train Labels - Provider x PotentialFraud")
train_labels.show(5)
train_labels.groupBy("PotentialFraud").count().show()
print(f"Total providers in training set: {train_labels.count():,}")

---

## Section 3 – Data Merging Pipeline

```
Inpatient + Outpatient  →  UNION (+ claim_type flag)
       ↓
+ Beneficiary           →  LEFT JOIN on BeneID
       ↓
+ Train Labels          →  LEFT JOIN on Provider
       ↓
     final_df
```

> **Note:** PotentialFraud is a _provider-level_ label. Every claim from a flagged provider inherits `PotentialFraud = Yes`.


In [0]:
# ============================================================
# CELL 9 – Build the merged modelling table
# ============================================================
inpatient = inpatient_raw.withColumn("claim_type", lit("inpatient"))
outpatient = outpatient_raw.withColumn("claim_type", lit("outpatient"))

claims_df = inpatient.unionByName(outpatient, allowMissingColumns=True)
print(
    f"After UNION  → {claims_df.count():,} rows  |  {len(claims_df.columns)} columns")

# LEFT JOIN Beneficiary on BeneID (keep all claims, bene data may be missing)
claims_bene_df = claims_df.join(beneficiary_raw, on="BeneID", how="left")
print(
    f"After +Bene  → {claims_bene_df.count():,} rows  |  {len(claims_bene_df.columns)} columns")

# INNER JOIN Labels on Provider — only keep claims from labeled providers
final_df = claims_bene_df.join(train_labels, on="Provider", how="inner")
print(
    f"After +Label → {final_df.count():,} rows  |  {len(final_df.columns)} columns")
print(f"\nFinal schema column count: {len(final_df.columns)}")
final_df.printSchema()

---

## Section 4 – Missing Value Analysis

Quantify null / missing values across every column in the merged table.


In [0]:
# Replace 'NA' and empty strings with nulls for all string columns
for c, dtype in final_df.dtypes:
    if dtype == "string":
        final_df = final_df.withColumn(
            c,
            when(
                (col(c) == "NA") | (col(c) == ""),
                None
            ).otherwise(col(c))
        )

total_rows = final_df.count()
dtypes_dict = dict(final_df.dtypes)

missing_exprs = []
for c in final_df.columns:
    dtype = dtypes_dict[c]
    if dtype in ("double", "float"):
        null_check = col(c).isNull() | isnan(col(c))
    else:
        null_check = col(c).isNull()
    missing_exprs.append(
        _sum(when(null_check, 1).otherwise(0)).alias(c)
    )

missing_counts = final_df.select(missing_exprs).toPandas().T
missing_counts.columns = ["missing_count"]
missing_counts["missing_pct"] = (
    missing_counts["missing_count"] / total_rows * 100
).round(2)
missing_counts = missing_counts.sort_values("missing_pct", ascending=False)

print(f"Total rows: {total_rows:,}\n")
print(missing_counts[missing_counts["missing_pct"] > 0].to_string())

# Visualise top 20 missing columns
top_missing = missing_counts[missing_counts["missing_pct"] > 0].head(20)
if len(top_missing) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top_missing.index, top_missing["missing_pct"], color="salmon")
    ax.set_xlabel("Missing %")
    ax.set_title("Top 20 Columns by Missing-Value Percentage")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

---

## Section 5 – Target Variable & Class Imbalance

Analyse the distribution of `PotentialFraud` across claims.


In [0]:
# How many providers are in Train.csv total?
train_df = spark.read.csv(TRAIN_LABEL_PATH, header=True, inferSchema=True)
print("Providers in Train.csv:", train_df.select("Provider").distinct().count())

# How many providers are in claims but missing from Train?
claims_providers = final_df.select("Provider").distinct()
train_providers = train_df.select("Provider").distinct()

unmatched = claims_providers.subtract(train_providers)
print("Claim providers not in Train.csv:", unmatched.count())

In [0]:
# ============================================================
# CELL 11 – Class distribution
# ============================================================
class_dist = final_df.groupBy("PotentialFraud").count().orderBy(
    "PotentialFraud").toPandas()
class_dist["pct"] = (class_dist["count"] /
                     class_dist["count"].sum() * 100).round(2)
print("PotentialFraud Distribution:")
print(class_dist.to_string(index=False))

fraud_pct = class_dist.loc[class_dist["PotentialFraud"]
                           == "Yes", "pct"].values[0]
print(
    f"\nImbalance ratio ≈ {100 - fraud_pct:.1f}% No  /  {fraud_pct:.1f}% Yes")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
sns.barplot(data=class_dist, x="PotentialFraud", y="count",
            ax=axes[0], palette=["#2ecc71", "#e74c3c"])
axes[0].set_title("Claim Count by Fraud Label")
axes[0].set_ylabel("Count")
for i, row in class_dist.iterrows():
    axes[0].text(i, row["count"] + 500,
                 f'{row["pct"]}%', ha="center", fontweight="bold")

# Pie chart
axes[1].pie(class_dist["count"], labels=class_dist["PotentialFraud"],
            autopct="%1.1f%%", colors=["#2ecc71", "#e74c3c"], startangle=90)
axes[1].set_title("Class Proportion")

plt.suptitle(
    "Target Variable: PotentialFraud (Provider-Level Proxy for Audit Risk)", fontweight="bold")
plt.tight_layout()
plt.show()

print("\n⚠ Strategy: We will NOT use SMOTE (healthcare codes are categorical).")
print("  → Algorithmic Class Weights + Probability Threshold Tuning (see Day 3 notebook).")

---

## Section 6 – Numerical Feature Distributions

Explore claim amounts, deductibles, and reimbursement patterns.


In [0]:
# ============================================================
# CELL 12 – Key numerical feature distributions
# ============================================================
num_cols = ["InscClaimAmtReimbursed", "DeductibleAmtPaid",
            "IPAnnualReimbursementAmt", "IPAnnualDeductibleAmt",
            "OPAnnualReimbursementAmt", "OPAnnualDeductibleAmt"]

# Filter to columns that actually exist in the dataframe
num_cols = [c for c in num_cols if c in final_df.columns]

print("Numerical Summary Statistics:")
final_df.select(num_cols).describe().show()

# Distribution by Fraud label
pdf = final_df.select(num_cols + ["PotentialFraud"]).toPandas()
pdf[num_cols] = pdf[num_cols].apply(pd.to_numeric, errors="coerce")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col_name in zip(axes.flatten(), num_cols):
    sns.boxplot(data=pdf, x="PotentialFraud", y=col_name, ax=ax,
                palette=["#2ecc71", "#e74c3c"], showfliers=False)
    ax.set_title(col_name, fontsize=10)
for ax in axes.flatten()[len(num_cols):]:
    ax.set_visible(False)
plt.suptitle(
    "Numerical Features by Fraud Label (outliers hidden for clarity)", fontweight="bold")
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# CELL 13 – Claim amount histograms
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col_name in zip(axes, ["InscClaimAmtReimbursed", "DeductibleAmtPaid"]):
    if col_name not in final_df.columns:
        continue
    sample_pdf = final_df.select(col_name, "PotentialFraud").toPandas()
    sample_pdf[col_name] = pd.to_numeric(sample_pdf[col_name], errors="coerce")
    for label, color in [("No", "#2ecc71"), ("Yes", "#e74c3c")]:
        subset = sample_pdf[sample_pdf["PotentialFraud"]
                            == label][col_name].dropna()
        ax.hist(subset, bins=50, alpha=0.6,
                label=f"Fraud={label}", color=color, density=True, zorder=2 if label == "Yes" else 1)
    ax.set_title(f"{col_name} Distribution")
    ax.set_xlabel(col_name)
    ax.set_ylabel("Density")
    ax.legend()

plt.suptitle("Claim Amount Distributions: Fraud vs Non-Fraud",
             fontweight="bold")
plt.tight_layout()
plt.show()

---

## Section 7 – Claim Type Analysis

Compare inpatient vs outpatient claims across fraud labels.


In [0]:
# ============================================================
# CELL 14 – Claim type breakdown
# ============================================================
ct_dist = final_df.groupBy("claim_type", "PotentialFraud").count().orderBy(
    "claim_type", "PotentialFraud").toPandas()
print("Claims by Type × Fraud Label:")
print(ct_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ct_pivot = ct_dist.pivot(
    index="claim_type", columns="PotentialFraud", values="count").fillna(0)
ct_pivot.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"])
ax.set_title("Claim Type × Fraud Label")
ax.set_ylabel("Count")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Fraud rate by claim type
print("\nFraud Rate by Claim Type:")
ct_rate = final_df.groupBy("claim_type").agg(
    count("*").alias("total"),
    _sum(when(col("PotentialFraud") == "Yes", 1).otherwise(0)).alias("fraud_count")
).withColumn("fraud_rate_pct", _round(col("fraud_count") / col("total") * 100, 2))
ct_rate.show()

---

## Section 8 – Chronic Conditions vs Fraud

Analyse which of the 11 chronic condition flags correlate with anomalous providers.


In [0]:
# ============================================================
# CELL 15 – Chronic condition prevalence by fraud label
# ============================================================
chronic_cols = [c for c in final_df.columns if c.startswith("ChronicCond_")]
print(f"Chronic condition columns found: {len(chronic_cols)}")

# For each chronic condition, compute the prevalence (value=1 means YES in some datasets, value=2 in others)
# The Kaggle dataset uses 1=Yes and 2=No for chronic conditions
chronic_data = []
for cc in chronic_cols:
    fraud_rate = final_df.filter(col(cc) == 1).groupBy(
        "PotentialFraud").count().toPandas()
    total = fraud_rate["count"].sum()
    yes_count = fraud_rate.loc[fraud_rate["PotentialFraud"]
                               == "Yes", "count"].values
    yes_pct = (yes_count[0] / total * 100) if len(yes_count) > 0 else 0
    chronic_data.append({"condition": cc.replace(
        "ChronicCond_", ""), "fraud_pct_among_positive": round(yes_pct, 2)})

chronic_pdf = pd.DataFrame(chronic_data).sort_values(
    "fraud_pct_among_positive", ascending=False)
print(chronic_pdf.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=chronic_pdf, x="fraud_pct_among_positive",
            y="condition", palette="Reds_r", ax=ax)
ax.set_xlabel("Fraud % Among Patients With This Condition")
ax.set_title("Chronic Condition Prevalence Among Fraud-Labelled Claims")
plt.tight_layout()
plt.show()

---

## Section 9 – Provider-Level Analysis

Since the fraud label is at the provider level, analyse provider-level patterns.


In [0]:
# ============================================================
# CELL 16 – Provider-level claim volumes & fraud
# ============================================================
provider_stats = final_df.groupBy("Provider", "PotentialFraud").agg(
    count("*").alias("claim_count"),
    countDistinct("BeneID").alias("unique_patients"),
    countDistinct("AttendingPhysician").alias("unique_physicians"),
    avg("InscClaimAmtReimbursed").alias("avg_claim_amt")
).orderBy(desc("claim_count"))

print("Top 15 Providers by Claim Volume:")
provider_stats.show(15, truncate=False)

# Distribution of claims per provider
prov_pdf = provider_stats.select(
    "Provider", "PotentialFraud", "claim_count").toPandas()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=prov_pdf, x="claim_count", hue="PotentialFraud", bins=40,
             palette=["#2ecc71", "#e74c3c"], ax=axes[0], kde=True)
axes[0].set_title("Distribution of Claim Count per Provider")
axes[0].set_xlabel("Number of Claims")

sns.boxplot(data=prov_pdf, x="PotentialFraud", y="claim_count",
            palette=["#2ecc71", "#e74c3c"], ax=axes[1], showfliers=False)
axes[1].set_title("Claims per Provider by Fraud Label")

plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# CELL 17 – Average claim amount by fraud label at provider level
# ============================================================
prov_amt = provider_stats.select(
    "PotentialFraud", "avg_claim_amt", "unique_patients").toPandas()
prov_amt["avg_claim_amt"] = pd.to_numeric(
    prov_amt["avg_claim_amt"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=prov_amt, x="PotentialFraud", y="avg_claim_amt",
            palette=["#2ecc71", "#e74c3c"], ax=axes[0], showfliers=False)
axes[0].set_title("Avg Claim Amount per Provider by Fraud Label")

sns.boxplot(data=prov_amt, x="PotentialFraud", y="unique_patients",
            palette=["#2ecc71", "#e74c3c"], ax=axes[1], showfliers=False)
axes[1].set_title("Unique Patients per Provider by Fraud Label")
plt.tight_layout()
plt.show()

---

## Section 10 – Diagnosis & Procedure Codes

Identify the most frequent diagnosis and procedure codes, especially among fraud-labelled claims.


In [0]:
# ============================================================
# CELL 18 – Top diagnosis codes
# ============================================================
from pyspark.sql import DataFrame
from functools import reduce
diag_cols = [c for c in final_df.columns if c.startswith("ClmDiagnosisCode_")]
print(f"Diagnosis code columns: {diag_cols}")

# Melt all diagnosis codes into a single column for frequency analysis

diag_dfs = []
for dc in diag_cols[:5]:  # Limit to first 5 for clarity
    diag_dfs.append(
        final_df.select(col(dc).alias("DiagnosisCode"), "PotentialFraud")
                .filter(col("DiagnosisCode").isNotNull())
    )

all_diag = reduce(DataFrame.unionAll, diag_dfs)
top_diag = all_diag.groupBy("DiagnosisCode").count().orderBy(
    desc("count")).limit(20).toPandas()

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_diag, x="count",
            y="DiagnosisCode", palette="viridis", ax=ax)
ax.set_title("Top 20 Most Frequent Diagnosis Codes (Codes 1-5)")
ax.set_xlabel("Frequency")
plt.tight_layout()
plt.show()

# Top diagnosis codes among fraud vs non-fraud
print("\nTop 10 Diagnosis Codes Among FRAUD Claims:")
all_diag.filter(col("PotentialFraud") == "Yes").groupBy(
    "DiagnosisCode")     .count().orderBy(desc("count")).show(10)

In [0]:
# ============================================================
# CELL 19 – Top procedure codes
# ============================================================
proc_cols = [c for c in final_df.columns if c.startswith("ClmProcedureCode_")]
print(f"Procedure code columns: {proc_cols}")

if proc_cols:
    proc_dfs = []
    for pc in proc_cols:
        proc_dfs.append(
            final_df.select(col(pc).alias("ProcedureCode"), "PotentialFraud")
                    .filter(col("ProcedureCode").isNotNull())
        )
    all_proc = reduce(DataFrame.unionAll, proc_dfs)
    top_proc = all_proc.groupBy("ProcedureCode").count().orderBy(
        desc("count")).limit(15).toPandas()

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=top_proc, x="count",
                y="ProcedureCode", palette="magma", ax=ax)
    ax.set_title("Top 15 Most Frequent Procedure Codes")
    ax.set_xlabel("Frequency")
    plt.tight_layout()
    plt.show()

---

## Section 11 – Temporal Patterns

Analyse claim submission trends over time and admission durations.


In [0]:
# ============================================================
# CELL 20 – Claim date trends
# ============================================================
temporal_df = final_df.withColumn("ClaimStartDt_d", to_date(col("ClaimStartDt"), "yyyy-MM-dd"))                       .withColumn(
    "claim_month", month("ClaimStartDt_d"))                       .withColumn("claim_year", year("ClaimStartDt_d"))

monthly = temporal_df.groupBy("claim_year", "claim_month", "PotentialFraud")     .count(
).orderBy("claim_year", "claim_month").toPandas()
monthly["period"] = monthly["claim_year"].astype(
    str) + "-" + monthly["claim_month"].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(14, 5))
for label, color in [("No", "#2ecc71"), ("Yes", "#e74c3c")]:
    subset = monthly[monthly["PotentialFraud"] == label]
    ax.plot(subset["period"], subset["count"], marker="o",
            label=f"Fraud={label}", color=color, linewidth=2)
ax.set_title("Monthly Claim Volume by Fraud Label")
ax.set_xlabel("Month")
ax.set_ylabel("Claim Count")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# CELL 21 – Inpatient admission duration analysis
# ============================================================
admit_df = final_df.filter(col("claim_type") == "inpatient")     .withColumn("AdmissionDt_d", to_date(col("AdmissionDt"), "yyyy-MM-dd"))     .withColumn(
    "DischargeDt_d", to_date(col("DischargeDt"), "yyyy-MM-dd"))     .withColumn("stay_days", datediff(col("DischargeDt_d"), col("AdmissionDt_d")))

print("Admission Duration by Fraud Label:")
admit_df.groupBy("PotentialFraud").agg(
    avg("stay_days").alias("avg_stay"),
    _min("stay_days").alias("min_stay"),
    _max("stay_days").alias("max_stay")
).show()

admit_pdf = admit_df.select("stay_days", "PotentialFraud").toPandas()
admit_pdf["stay_days"] = pd.to_numeric(admit_pdf["stay_days"], errors="coerce")

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=admit_pdf, x="stay_days", hue="PotentialFraud", bins=30,
             palette=["#2ecc71", "#e74c3c"], kde=True, ax=ax)
ax.set_title("Inpatient Admission Duration Distribution by Fraud Label")
ax.set_xlabel("Stay Duration (days)")
plt.tight_layout()
plt.show()

---

## Section 12 – Correlation Analysis

Compute a correlation matrix for key numerical features.


In [0]:
# ============================================================
# CELL 22 – Correlation heatmap
# ============================================================
# Binary-encode PotentialFraud for correlation
corr_df = final_df.withColumn("Fraud_Binary", when(
    col("PotentialFraud") == "Yes", 1).otherwise(0))

corr_cols = ["InscClaimAmtReimbursed", "DeductibleAmtPaid",
             "IPAnnualReimbursementAmt", "IPAnnualDeductibleAmt",
             "OPAnnualReimbursementAmt", "OPAnnualDeductibleAmt",
             "Fraud_Binary"]
corr_cols = [c for c in corr_cols if c in corr_df.columns]

corr_pdf = corr_df.select(corr_cols).toPandas().apply(
    pd.to_numeric, errors="coerce")
corr_matrix = corr_pdf.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Correlation Matrix – Key Numerical Features vs Fraud Label")
plt.tight_layout()
plt.show()

---

## Section 13 – Key Findings & Observations

### 1. Dataset Structure

- **Inpatient claims** contain admission/discharge dates and procedure codes; **outpatient** claims do not.
- After UNION, missing columns (`AdmissionDt`, `DischargeDt`) are naturally null for outpatient rows — this is correct and meaningful.
- The **Beneficiary** table adds 11 chronic condition flags and demographic data.

### 2. Class Imbalance

- PotentialFraud ≈ **10% Yes / 90% No** — this confirms the architecture plan's recommendation.
- **SMOTE** is not suitable due to categorical healthcare codes and PySpark scalability concerns.
- **Strategy:** Algorithmic Class Weights + Probability Threshold Tuning (to be implemented in Day 3).

### 3. Patterns in Anomalous Claims

- Fraud-labelled providers tend to have **higher claim volumes** and **more unique patients** per provider.
- Certain **diagnosis codes** appear with disproportionate frequency among fraud-labelled claims.
- **Inpatient admission durations** may differ between fraud and non-fraud providers.
- Reimbursement and deductible amounts show distinct distribution patterns for fraud vs non-fraud.

### 4. Missing Values

- Outpatient-specific columns (`AdmissionDt`, `DischargeDt`) will have high null rates — this is expected from the UNION.
- Procedure codes have significant nulls — many claims do not have all 6 procedure slots filled.
- Some diagnosis code slots (7-10) also have high null rates.

### 5. Next Steps (Day 2 – Feature Engineering)

- Handle remaining missing values (imputation vs indicators)
- Remove duplicates if any
- Engineer new features: claim category, provider frequency, admission duration bins
- Compute algorithmic class weights for the `PotentialFraud` label
- Create the clean Delta Table for modelling
